# Treinamento de Modelo YOLOv8 para Detecção de Jogadores

Projeto de treinamento usando YOLOv8, com preparação automática de datasets e configuração do modelo.

## 1. Instalar dependências

In [ ]:
!pip install ultralytics

## 2. Imports

In [ ]:
from ultralytics import YOLO
import yaml
import os

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from PIL import Image
from IPython.display import display, Image as DispImage

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Defina o diretório do dataset
dataset_dir = './data/yolo_dataset'

dir_train = os.path.join(dataset_dir, 'images', 'train')
dir_val = os.path.join(dataset_dir, 'images', 'val')
train_images = [f for f in os.listdir(dir_train) if f.lower().endswith(('jpg','png','jpeg'))]
val_images = [f for f in os.listdir(dir_val) if f.lower().endswith(('jpg','png','jpeg'))]

counts = pd.DataFrame({
    'split': ['train', 'val'],
    'image_count': [len(train_images), len(val_images)]
})

# Exibir a tabela
display(counts)

# Plotar distribuição de imagens
plt.figure()
sns.barplot(data=counts, x='split', y='image_count')
plt.title('Número de Imagens por Split')
plt.show()

# Mostrar algumas imagens de exemplo
plt.figure(figsize=(8,4))
for i, img_name in enumerate(train_images[:4]):
    img = Image.open(os.path.join(dir_train, img_name))
    plt.subplot(1,4,i+1)
    plt.imshow(img)
    plt.axis('off')
plt.suptitle('Exemplos de Imagens de Treino')
plt.show()

## 4. Preparar `data.yaml` automaticamente e criar classes

In [ ]:
classes_path = os.path.join(dataset_dir, 'classes.txt')
data_yaml_path = 'data.yaml'

if not os.path.exists(data_yaml_path):
    with open(classes_path, 'r') as f:
        class_names = [name.strip() for name in f.readlines() if name.strip()]
    yaml_content = {
        'path': dataset_dir,
        'train': 'images/train',
        'val': 'images/val',
        'names': {i: name for i, name in enumerate(class_names)}
    }
    with open(data_yaml_path, 'w') as f:
        yaml.dump(yaml_content, f, default_flow_style=False)
    print("Arquivo data.yaml criado com sucesso!")

## 5. Função de treinamento via API Python

In [ ]:
def train_yolov8(data_yaml, model_size='m', epochs=50, imgsz=640, batch=16):
    model = YOLO(f'yolov8{model_size}.pt')
    results = model.train(
        data=data_yaml,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        name=f'football_yolov8{model_size}',
        save=True,
        save_period=10,
        val=True
    )
    return results

## 6. Treinar o modelo

In [ ]:
results = train_yolov8(data_yaml=data_yaml_path, model_size='m', epochs=50, imgsz=640, batch=16)

## 7. Visualizar resultados e métricas

In [ ]:
display(DispImage(filename='runs/detect/football_yolov8m/results.png'))
display(DispImage(filename='runs/detect/football_yolov8m/confusion_matrix.png'))

## 8. Plotar curvas de métricas com pandas e matplotlib

In [ ]:
df = pd.read_csv('runs/detect/football_yolov8m/results.csv')
df_plot = df.set_index('epoch')
plt.figure()
plt.plot(df_plot['train/box_loss'], label='train/box_loss')
plt.plot(df_plot['val/box_loss'], label='val/box_loss')
plt.plot(df_plot['train/cls_loss'], label='train/cls_loss')
plt.plot(df_plot['val/cls_loss'], label='val/cls_loss')
plt.title('Loss de Treino e Validação')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.figure()
plt.plot(df_plot['metrics/precision(B)'], label='Precision')
plt.plot(df_plot['metrics/recall(B)'], label='Recall')
plt.plot(df_plot['metrics/mAP50(B)'], label='mAP50')
plt.title('Métricas de Detecção por Epoch')
plt.xlabel('Epoch')
plt.ylabel('Valor')
plt.legend()
plt.show()

## 9. Alternativa: Treinar diretamente via CLI

In [ ]:
# !yolo task=detect mode=train model=yolov8m.pt data=data.yaml epochs=50 imgsz=640